In [1]:
pip install ultralytics opencv-python

Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 24.3.1 -> 25.3
[notice] To update, run: python.exe -m pip install --upgrade pip


In [2]:
import base64
from IPython.display import HTML, display

def show_mp4(path, width=640):
    """
    Reads an MP4 from disk and embeds it as an HTML5 video in Colab.

    Args:
        path (str): path to the .mp4 file
        width (int): display width in pixels
    """
    # Read & encode
    with open(path, 'rb') as f:
        mp4_bytes = f.read()
    b64 = base64.b64encode(mp4_bytes).decode()
    # Build HTML
    html = f'''
    <video width="{width}" controls autoplay loop muted playsinline>
      <source src="data:video/mp4;base64,{b64}" type="video/mp4">
      Your browser does not support HTML5 video.
    </video>
    '''
    display(HTML(html))

In [3]:
show_mp4("catalogue/planes.mp4")

FileNotFoundError: [Errno 2] No such file or directory: 'catalogue/planes.mp4'

In [ ]:
import cv2
import random
from ultralytics import YOLO
import numpy as np

# --- Configuration Globale ---
# NOTE: Remplacez 'yolov8x.pt' par une version plus rapide (ex: 'yolov8s.pt')
# si vous rencontrez des problèmes de performance sans GPU.
MODEL_PATH = r"C:\Studies\S7\AI\drone_detection_project\step1_detection\runs_finetune\content\runs\detect\finetune_drone_colab\weights\best.pt"\n

# Chargez le modèle YOLOv8 (une seule fois)
model = YOLO(MODEL_PATH)
# Définition des classes cibles (Exemple : 4 pour "airplane" si vous utilisez un modèle COCO)
# Si vous entraînez votre propre modèle, utilisez l'ID de votre classe "drone/objet volant".
TARGET_CLASSES = [4]

def bytetrack_and_extract(video_path, output_path="output_tracked.mp4", target_classes=None):
    """
    Exécute le tracking ByteTrack sur une vidéo, génère la vidéo de sortie avec
    les boîtes englobantes et retourne une liste des données de trajectoire structurées.
    """

    # --- 1. Préparation de la Vidéo ---
    cap = cv2.VideoCapture(video_path)
    if not cap.isOpened():
        print(f"Erreur : Impossible d'ouvrir le fichier vidéo à l'emplacement {video_path}")
        return []

    fps = cap.get(cv2.CAP_PROP_FPS)
    width = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))
    height = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))

    # Initialisation de l'écriture vidéo
    fourcc = cv2.VideoWriter_fourcc(*'mp4v') # Codec recommandé pour la compatibilité (ex: mp4v)
    out = cv2.VideoWriter(output_path, fourcc, fps, (width, height))

    # Dictionnaire pour stocker des couleurs uniques par ID de piste pour la visualisation
    track_id_colors = {}

    # Nouvelle liste pour stocker toutes les données de trajectoire (votre structure de données)
    all_tracks_data = []

    print(f"Démarrage du tracking sur la vidéo ({width}x{height} @ {fps:.2f} FPS)...")

    # --- 2. Exécution du Tracking avec ByteTrack ---
    # Nous utilisons la source directement dans model.track
    results = model.track(
        source=video_path,
        tracker="bytetrack.yaml", # Assurez-vous que ce fichier est dans le bon chemin
        persist=True,
        stream=True,
        classes=target_classes,
        conf=0.2 # Seuil bas pour capturer les "bytes" de ByteTrack
    )

    # --- 3. Boucle de Traitement Trame par Trame ---
    for frame_id, result in enumerate(results):

        # Récupère l'image originale non annotée (numpy array)
        frame = result.orig_img.copy()

        # Vérification essentielle : le tracker doit avoir assigné des IDs
        if result.boxes is not None and result.boxes.id is not None:

            # Extraction des données du GPU vers le CPU et conversion en listes Python
            track_ids = result.boxes.id.int().cpu().tolist()
            bboxes = result.boxes.xyxy.cpu().tolist()
            class_ids = result.boxes.cls.int().cpu().tolist()

            # Traitement de chaque objet suivi dans la trame
            for track_id, bbox, cls_id in zip(track_ids, bboxes, class_ids):

                # Filtrage optionnel des classes (si target_classes est défini)
                if target_classes and cls_id not in target_classes:
                    continue

                # Coordonnées du Bounding Box
                x1, y1, x2, y2 = map(int, bbox)

                # --- A. Extraction des Données Structurées ---
                # Calcul des coordonnées centrales (point clé pour la trajectoire)
                x_center = (x1 + x2) / 2
                y_center = (y1 + y2) / 2

                # Stockage du point de trajectoire
                track_point = {
                    "track_id": track_id,
                    "frame_id": frame_id,
                    "x_center": x_center,
                    "y_center": y_center
                }
                all_tracks_data.append(track_point)

                # --- B. Visualisation (Dessin sur la Trame) ---

                # Assignation/Récupération de couleur unique par ID
                if track_id not in track_id_colors:
                    track_id_colors[track_id] = (random.randint(0, 255), random.randint(0, 255), random.randint(0, 255))
                color = track_id_colors[track_id]

                # Dessiner la boîte englobante
                cv2.rectangle(frame, (x1, y1), (x2, y2), color, 3)

                # Afficher l'ID
                text = f"ID: {track_id}"
                cv2.putText(frame, text,
                            (x1, y1 - 10), # Positionnement au-dessus de la boîte
                            cv2.FONT_HERSHEY_SIMPLEX, 1.0, color, 2)

            # Enregistrement de la trame annotée
            out.write(frame)
            print(f"Traitement en cours - Frame {frame_id+1}/{int(cap.get(cv2.CAP_PROP_FRAME_COUNT))}", end='\r')

    # --- 4. Finalisation ---
    cap.release()
    out.release()
    print(f"\n--- Terminé ---")
    print(f"Vidéo de sortie sauvée dans : '{output_path}'")
    print(f"Nombre total de points de trajectoire extraits : {len(all_tracks_data)}")

    return all_tracks_data

# --- Exemple d'Utilisation ---
VIDEO_INPUT = r"C:\Studies\S7\AI\drone_detection_project\drone_videos\drone_vid1.mp4"\n
TRACKS_OUTPUT_FILE = "tracking_data.csv"

# Exécuter la fonction et récupérer les données
trajectories = bytetrack_and_extract(VIDEO_INPUT, target_classes=TARGET_CLASSES)

# Si vous souhaitez stocker les données dans un fichier CSV (Recommandé pour l'analyse)
try:
    import pandas as pd
    df_trajectories = pd.DataFrame(trajectories)
    df_trajectories.to_csv(TRACKS_OUTPUT_FILE, index=False)
    print(f"Données de trajectoire stockées dans : {TRACKS_OUTPUT_FILE}")

except ImportError:
    print("\nAttention : La librairie Pandas n'est pas installée. Impossible de sauvegarder en CSV.")
    print("Les 5 premiers points extraits :")
    print(trajectories[:5])

Démarrage du tracking sur la vidéo (1920x1080 @ 29.97 FPS)...

video 1/1 (frame 1/148) /content/catalogue/planes.mp4: 384x640 4 airplanes, 2882.9ms
video 1/1 (frame 2/148) /content/catalogue/planes.mp4: 384x640 4 airplanes, 2893.7ms
video 1/1 (frame 3/148) /content/catalogue/planes.mp4: 384x640 4 airplanes, 3986.3ms
video 1/1 (frame 4/148) /content/catalogue/planes.mp4: 384x640 4 airplanes, 2888.9ms
video 1/1 (frame 5/148) /content/catalogue/planes.mp4: 384x640 4 airplanes, 2897.6ms
video 1/1 (frame 6/148) /content/catalogue/planes.mp4: 384x640 4 airplanes, 3009.3ms
video 1/1 (frame 7/148) /content/catalogue/planes.mp4: 384x640 4 airplanes, 4743.4ms
video 1/1 (frame 8/148) /content/catalogue/planes.mp4: 384x640 4 airplanes, 3012.2ms
video 1/1 (frame 9/148) /content/catalogue/planes.mp4: 384x640 4 airplanes, 3551.9ms
video 1/1 (frame 10/148) /content/catalogue/planes.mp4: 384x640 4 airplanes, 2846.5ms
video 1/1 (frame 11/148) /content/catalogue/planes.mp4: 384x640 4 airplanes, 3951.5ms
